### Schiebeparkplatz – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021), 1. Runde, Aufgabe 1*

Auf dem „Schiebeparkplatz“ in Tübingen darf man Wagen auch quer vor den normalen Plätzen abstellen. Quer stehende Wagen
können vorsichtig vorwärts oder rückwärts verschoben werden, um einem blockierten Wagen die Ausfahrt zu ermöglichen.

Das Bild zeigt eine Situation mit einer einzelnen „Parkreihe“ (hier als Text; `_` ist ein freier Platz, ein quer stehender Wagen belegt zwei Plätze):

```
A B C D E F G      normale Wagen
_ _ H H _ I I      quer stehende Wagen H und I
```

- Damit die Wagen A, B oder E ausfahren können, muss kein anderer Wagen verschoben werden.
- Damit die Wagen C, D oder G ausfahren können, muss jeweils mindestens ein anderer Wagen verschoben werden.
- Damit der Wagen F ausfahren kann, müssen hingegen zuerst Wagen H und dann Wagen I verschoben werden.

#### Aufgabe 1

Schreibe ein Programm, das eine Situation auf einem Schiebeparkplatz mit einer Parkreihe einliest und für jeden Wagen auf
einem normalen Platz bestimmt, welche anderen Wagen wie verschoben werden müssen, damit er ausfahren kann. Dabei sollen
möglichst wenige Wagen verschoben werden.

Die Ausgabe des Programms für das obige Beispiel wäre:

```
A:
B:
C: H 1 rechts
D: H 1 links
E:
F: H 1 links, I 2 links
G: I 1 links
```

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/parkplatz0.txt` bis `parkplatz5.txt`)

- 1. Zeile: Bezeichnung des ersten und letzten Autos auf den normalen Plätzen (z. B. `A G`)
- 2. Zeile: Anzahl n der quer stehenden Autos
- je quer stehendes Auto: Bezeichnung und Position p (ab 0 von links); es belegt die Positionen p und p + 1

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zwei Möglichkeiten durchrechnen und vergleichen:** Ein Wagen kann sein Hindernis nach links oder nach rechts verlassen. Man berechnet beide
  Möglichkeiten und nimmt die bessere.
- **Rekursion:** Wenn ein quer stehender Wagen sich verschiebt, stößt er vielleicht an den nächsten quer stehenden Wagen, der dann
  ebenfalls weichen muss, und so weiter. „Wagen verschieben“ ruft dann sich selbst für den Nachbarn auf.
- **Vergleichen mit Tupeln:** Python vergleicht Tupel Eintrag für Eintrag. `(2, 3) < (3, 1)` ist wahr. So lässt sich „erst möglichst wenige Wagen,
  bei Gleichstand möglichst kurze Wege“ in einem einzigen Vergleich ausdrücken.

**Nützliche Funktionen und Bibliotheken**
- `ord('C') - ord('A')` ergibt die Nummer eines Buchstabens (2), `chr(ord('A') + 2)` den Buchstaben dazu.
- `sum(...)` für Listen von Zahlen.

## Die Idee

Die Plätze sind von links nach rechts mit `0, 1, 2, ...` nummeriert. Der normale Wagen mit dem Buchstaben `A` steht auf Platz 0, `B` auf Platz 1 und so weiter.
Ein quer stehender Wagen, der bei `p` steht, belegt die Plätze `p` und `p + 1`.

**Wann ist ein Wagen blockiert?** Wenn ein quer stehender Wagen vor seinem Platz `x` steht, also `p == x` oder `p + 1 == x` gilt.
Dann gibt es nur zwei Auswege:

- **nach links**: Der Querwagen schiebt sich so weit nach links, dass er mit dem rechten Ende auf `x - 1` steht, also neue Position `x - 2`.
  Das ist die *Schlüsselposition* aus den Lösungshinweisen. Weniger weit reicht nicht, weiter bringt nichts.
- **nach rechts**: neue Position `x + 1`.

Beides geht nur, wenn der Querwagen auf dem Parkplatz bleibt (Position von `0` bis `Anzahl Plätze - 2`).

**Die Kettenreaktion:** Wenn der Querwagen auf seiner neuen Position auf einen anderen Querwagen stößt, muss der ebenfalls ausweichen,
und zwar in dieselbe Richtung und *gerade so weit*, dass die beiden nebeneinander passen (der Nachbar bekommt dann die Position `neue Position - 2`
bei Richtung links, `neue Position + 2` bei rechts). Das kann sich über mehrere Wagen fortsetzen. Stößt die Kette am Rand des Parkplatzes an, geht diese Richtung nicht.

Es sind keine anderen Wagen als die der Kette beteiligt, und jeder Wagen der Kette muss mindestens so weit wie berechnet ausweichen.
Deshalb ist jede Richtung für sich schon optimal: Kein Wagen wird unnötig bewegt, keiner weiter als nötig. Dann vergleichen wir links gegen rechts:
zuerst die **Anzahl** der verschobenen Wagen (das verlangt die Aufgabe), bei Gleichstand die **Summe der Wege**.

*Hinweis zur Reihenfolge der Ausgabe:* Die Wagen müssen in der Reihenfolge verschoben werden, in der wir sie ausgeben (bei links zuerst der am weitesten links stehende
Wagen der Kette, bei rechts der am weitesten rechts stehende). Sonst würden sich zwei Wagen im Weg stehen.

## Schritt 1: Einlesen

Wir lesen die Datei in drei Dinge ein: den ersten Buchstaben `erster`, die Anzahl der normalen Plätze `anzahl` und die Liste der Querwagen als
`(Position, Name)`, aufsteigend nach der Position sortiert. Die Sortierung ist wichtig, denn der linke und der rechte Nachbar eines Querwagens sind dann
der Eintrag davor bzw. dahinter in der Liste. Der Querwagen `H` auf Position 2 wird also zu `(2, 'H')`.

In [ ]:
def lies_parkplatz(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = [zeile.split() for zeile in f.read().splitlines() if zeile.strip()]
    erster, letzter = zeilen[0]
    anzahl = ord(letzter) - ord(erster) + 1
    n = int(zeilen[1][0])
    quer = [(int(p), name) for name, p in zeilen[2:2 + n]]
    quer.sort()
    return erster, anzahl, quer


erster, anzahl, quer = lies_parkplatz('beispieldaten/parkplatz0.txt')
print('erster Wagen:', erster, ' Plätze:', anzahl)
print('Querwagen:', quer)

erster Wagen: A  Plätze: 7
Querwagen: [(2, 'H'), (5, 'I')]


## Schritt 2: Ein Bild des Parkplatzes

Zur Kontrolle erzeugen wir die beiden Textzeilen aus der Aufgabe. In der unteren Zeile steht an jedem Platz entweder `_` oder der Buchstabe des Querwagens,
der diesen Platz belegt.

In [ ]:
def zeige(erster, anzahl, quer):
    oben = [chr(ord(erster) + i) for i in range(anzahl)]
    unten = ['_'] * anzahl
    for pos, name in quer:
        unten[pos] = name
        unten[pos + 1] = name
    print(' '.join(oben))
    print(' '.join(unten))


zeige(erster, anzahl, quer)

A B C D E F G
_ _ H H _ I I


## Schritt 3: Die Kette beim Schieben

`schiebe(quer, i, ziel, anzahl)` verschiebt den Querwagen mit Nummer `i` in der Liste auf die Position `ziel` und liefert die nötigen Verschiebungen als Liste von
Einträgen `(Name, Weg, Richtung)` in der Reihenfolge, in der sie ausgeführt werden müssen, oder `None`, wenn es nicht geht.

- Ist `ziel` kleiner als der aktuelle Platz, geht es nach **links**. Der linke Nachbar `i - 1` stört, wenn er mit seinem rechten Ende
  (`quer[i-1][0] + 1`) bis `ziel` oder weiter reicht. Dann muss er selbst nach links auf `ziel - 2`. Das ist der rekursive Aufruf.
  Seine Verschiebungen kommen **vor** unserer eigenen in die Liste.
- Ist `ziel` größer, geht es nach **rechts**, spiegelverkehrt: Der rechte Nachbar stört, wenn sein Platz kleiner als `ziel + 2` ist, und muss auf `ziel + 2`.
- Ragt der Wagen über den Rand hinaus (`ziel < 0` oder `ziel + 1 >= anzahl`), gibt es `None`, und das `None` reicht die Kette bis nach oben durch.

In [ ]:
def schiebe(quer, i, ziel, anzahl):
    pos, name = quer[i]
    if ziel < 0 or ziel + 1 >= anzahl:
        return None                              # ragt über den Rand hinaus
    if ziel < pos:                               # nach links
        liste = []
        if i > 0 and quer[i - 1][0] + 1 >= ziel:         # linker Nachbar im Weg
            liste = schiebe(quer, i - 1, ziel - 2, anzahl)
            if liste is None:
                return None
        return liste + [(name, pos - ziel, 'links')]
    else:                                        # nach rechts
        liste = []
        if i < len(quer) - 1 and quer[i + 1][0] < ziel + 2:   # rechter Nachbar im Weg
            liste = schiebe(quer, i + 1, ziel + 2, anzahl)
            if liste is None:
                return None
        return liste + [(name, ziel - pos, 'rechts')]


# Beispiel 0: H (Position 2) auf Position 1, I (Position 5) auf Position 3, I auf Position 6
print(schiebe(quer, 0, 1, anzahl))
print(schiebe(quer, 1, 3, anzahl))
print(schiebe(quer, 1, 6, anzahl))

[('H', 1, 'links')]
[('H', 1, 'links'), ('I', 2, 'links')]
None


Der zweite Aufruf zeigt die Kette: Wenn I (Position 5) auf Position 3 rutscht, stößt es an H (Position 2, belegt 2 und 3). H muss deshalb zuerst
um 1 nach links. Der dritte Aufruf scheitert, weil I dann bis Platz 7 reichen würde. Der Parkplatz hat nur die Plätze 0 bis 6.

## Schritt 4: Für einen Wagen die beste Lösung finden

Zu jedem normalen Wagen am Platz `x` suchen wir den Querwagen, der den Platz belegt (falls es einen gibt). Dann berechnen wir beide Möglichkeiten (links nach `x - 2`,
rechts nach `x + 1`), bewerten jede mit dem Tupel `(Anzahl Wagen, Summe der Wege)` und nehmen die kleinere.

In [ ]:
def blockierer(quer, x):
    for i in range(len(quer)):
        if quer[i][0] == x or quer[i][0] + 1 == x:
            return i
    return None


def bewertung(liste):
    return (len(liste), sum(weg for name, weg, richtung in liste))


def beste_loesung(quer, x, anzahl):
    i = blockierer(quer, x)
    if i is None:
        return []
    links = schiebe(quer, i, x - 2, anzahl)
    rechts = schiebe(quer, i, x + 1, anzahl)
    if links is None and rechts is None:
        return None                                   # keine Lösung
    if links is None:
        return rechts
    if rechts is None:
        return links
    if bewertung(rechts) < bewertung(links):
        return rechts
    return links


def text(liste):
    if liste is None:
        return 'nicht möglich'
    return ', '.join(f'{name} {weg} {richtung}' for name, weg, richtung in liste)


for x in range(anzahl):
    print(chr(ord(erster) + x) + ':', text(beste_loesung(quer, x, anzahl)))

A: 
B: 
C: H 1 rechts
D: H 1 links
E: 
F: H 1 links, I 2 links
G: I 1 links


## Schritt 5: Alles zusammen, mit Kontrolle

Damit sicher ist, dass die angegebenen Verschiebungen auch wirklich gehen, gibt es ein kleines **Kontrollprogramm** `pruefe`: Es führt die
Verschiebungen der Reihe nach aus. Vor jedem Schritt prüft es, dass der Wagen auf dem Parkplatz bleibt und dass er auf keinen anderen Querwagen trifft.
Am Ende muss der Platz des normalen Wagens frei sein. Außerdem darf jeder Wagen nur einmal vorkommen.

In [ ]:
def pruefe(quer, x, anzahl, liste):
    aktuell = {name: pos for pos, name in quer}
    namen = [name for name, weg, richtung in liste]
    ok = len(namen) == len(set(namen))
    for name, weg, richtung in liste:
        neu = aktuell[name] + (weg if richtung == 'rechts' else -weg)
        ok = ok and 0 <= neu and neu + 1 < anzahl
        for andere, pos in aktuell.items():              # kein Zusammenstoß mit anderen Querwagen
            if andere != name:
                ok = ok and (neu + 1 < pos or pos + 1 < neu)
        aktuell[name] = neu
    for name, pos in aktuell.items():                    # Platz x muss frei sein
        ok = ok and not (pos == x or pos + 1 == x)
    return ok


def loese(dateiname):
    erster, anzahl, quer = lies_parkplatz(dateiname)
    zeige(erster, anzahl, quer)
    for x in range(anzahl):
        liste = beste_loesung(quer, x, anzahl)
        if liste is not None:
            assert pruefe(quer, x, anzahl, liste)
        print(chr(ord(erster) + x) + ':', text(liste))


for nr in range(6):
    print('parkplatz' + str(nr) + '.txt')
    loese('beispieldaten/parkplatz' + str(nr) + '.txt')
    print()

parkplatz0.txt
A B C D E F G
_ _ H H _ I I
A: 
B: 
C: H 1 rechts
D: H 1 links
E: 
F: H 1 links, I 2 links
G: I 1 links

parkplatz1.txt
A B C D E F G H I J K L M N
_ O O P P _ Q Q _ _ R R _ _
A: 
B: P 1 rechts, O 1 rechts
C: O 1 links
D: P 1 rechts
E: O 1 links, P 1 links
F: 
G: Q 1 rechts
H: Q 1 links
I: 
J: 
K: R 1 rechts
L: R 1 links
M: 
N: 

parkplatz2.txt
A B C D E F G H I J K L M N
_ _ O O _ P P Q Q R R _ S S
A: 
B: 
C: O 1 rechts
D: O 1 links
E: 
F: O 1 links, P 2 links
G: P 1 links
H: R 1 rechts, Q 1 rechts
I: P 1 links, Q 1 links
J: R 1 rechts
K: P 1 links, Q 1 links, R 1 links
L: 
M: P 1 links, Q 1 links, R 1 links, S 2 links
N: S 1 links

parkplatz3.txt
A B C D E F G H I J K L M N
_ O O _ P P _ _ Q Q R R S S
A: 
B: O 1 rechts
C: O 1 links
D: 
E: P 1 rechts
F: P 1 links
G: 
H: 
I: Q 2 links
J: Q 1 links
K: Q 2 links, R 2 links
L: Q 1 links, R 1 links
M: Q 2 links, R 2 links, S 2 links
N: Q 1 links, R 1 links, S 1 links

parkplatz4.txt
A B C D E F G H I J K L M N O P
Q Q R R _ 

## Auswertung

Die Ausgaben stimmen in der Anzahl der verschobenen Wagen und in den Wegen mit den Beispielen der offiziellen Lösungshinweise überein (die Reihenfolge der Wagen in einer
Zeile darf beliebig sein). Einige Beobachtungen:

- `parkplatz0`: wie in der Aufgabe. F braucht zwei Verschiebungen: I muss um 2 nach links, und dafür vorher H um 1 nach links.
- `parkplatz2`, Wagen M: Die Kette ist mit vier Wagen am längsten (P, Q, R und S), und nur die Richtung links geht.
- `parkplatz5`, Wagen F: Hier ist ein Wagen um 2 nach rechts (Q) besser als zwei Wagen um je 1 nach links.
  Das verlangen auch die Bewertungskriterien: erst möglichst wenige Wagen, nicht die kürzeste Gesamtstrecke.
- `parkplatz4`: Q und R stehen ganz links vor A, B, C und D. Für A und B geht es nur nach rechts, und R muss mit Q weichen.

**Aufwand:** Pro normalem Wagen werden höchstens zwei Ketten durchlaufen, jede hat höchstens so viele Glieder, wie es Querwagen gibt. Das ist sehr schnell.

## Für Neugierige

**Warum ist die Kette wirklich optimal?** Steht der Querwagen `Q` am Platz `x` im Weg, muss er mit dem rechten Ende mindestens bis `x - 1` (oder mit dem linken
Ende mindestens bis `x + 1`) rücken, sonst bleibt der Platz belegt. Ein Wagen der Kette, der weiter ginge als nötig, würde nur weitere Wagen mitschieben, und
ein Wagen, der nicht in der Kette steht, wird nie berührt. Außerdem schiebt ein Wagen den Nachbarn nur gerade so weit, wie der Platz es verlangt. Also ist die Kette
in jeder der beiden Richtungen die kleinste mögliche Menge von Verschiebungen.

**Was ist mit Wagen, die in beide Richtungen nicht freikommen?** Dann liefert `beste_loesung` `None`, und das Programm schreibt „nicht möglich“. In den
Beispielen kommt das nicht vor, aber bei einem Parkplatz, der von Querwagen fast ganz ausgefüllt ist, wäre das möglich.

**MiniZinc?** Die Verschiebungen ergeben sich hier zwingend als Kette. Es gibt nur zwei Möglichkeiten pro Wagen zu vergleichen, aber nichts, was ein Solver suchen
müsste. Daher gibt es in diesem Notebook keinen MiniZinc-Abschnitt.